# pulsar — Tutorial

**pulsar** is a real-time hardware monitoring dashboard that runs in the terminal, designed for developers and researchers who need to track compute resources without leaving the command line.

It displays:
- Per-core CPU load (colour-coded progress bars) and live frequency per core
- Memory usage
- Disk I/O throughput (live MB/s)
- Network I/O throughput (live MB/s)
- A ranked list of the most resource-intensive processes (with in-dashboard kill support)
- A rotating fun-facts bar with a Pac-Man typing animation

This notebook shows how to use pulsar via its Python API and from the command line.

## 1. Installation


In [ ]:
# Install in editable mode from the project root — only needs to be run once
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    capture_output=True, text=True
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-300:])


## 2. Project Structure

```
pulsar/
├── pyproject.toml          # package config & dependency declarations
├── src/pulsar/
│   ├── __init__.py         # public API entry point
│   ├── cli.py              # command-line interface (click)
│   ├── collector.py        # hardware data collection (psutil)
│   ├── dashboard.py        # live TUI dashboard (rich.live)
│   ├── facts.py            # rotating fun-facts pool (local + online)
│   └── snapshot.py         # single-snapshot output (table / json)
└── tests/
```

Data flow:
```
cli.py
  ├── --once  →  collector.collect()  →  snapshot.print_table() / print_json()
  └── (live)  →  dashboard.run()  →  collector.collect() in a loop
                                   └── facts.get_fact()   in a loop
```

## 3. Python API — Calling the Collector Directly

### 3.1 Read Static System Information (call once at startup)


In [ ]:
from pulsar import get_system_info

info = get_system_info()
print(f"CPU model      : {info.cpu_model}")
print(f"Logical cores  : {info.cpu_cores}")
print(f"Current freq   : {info.cpu_freq_current} GHz  (max {info.cpu_freq_max} GHz)")
print(f"GPU            : {info.gpu_model}")
print(f"Total RAM      : {info.ram_total} GB")


### 3.2 Collect a Single Live Snapshot


In [ ]:
import time
from pulsar import collect

# First call initialises the disk I/O baseline (returning 0.0 MB/s is expected)
collect()
time.sleep(0.5)

# Second call produces a meaningful disk I/O delta
snap = collect(top_n=5)

print("=== CPU ===")
for i, pct in enumerate(snap.cpu_per_core):
    bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    print(f"  Core {i:>2}  {bar}  {pct:5.1f}%")
print(f"  Average  {snap.cpu_avg:.1f}%")

print("\n=== Memory ===")
print(f"  Used: {snap.mem_used} GB / Total: {snap.mem_total} GB  ({snap.mem_percent:.1f}%)")

print("\n=== Disk I/O ===")
print(f"  Read : {snap.disk_read_mbps:.2f} MB/s")
print(f"  Write: {snap.disk_write_mbps:.2f} MB/s")

print("\n=== Top 5 Processes ===")
print(f"  {'PID':>7}  {'Name':<24}  {'CPU%':>6}  {'MEM':>10}")
for p in snap.top_procs:
    mem_str = f"{p['mem_mb']} MB" if p['mem_mb'] < 1024 else f"{p['mem_mb']/1024:.2f} GB"
    print(f"  {p['pid']:>7}  {p['name']:<24}  {p['cpu']:>5.1f}%  {mem_str:>10}")


### 3.3 Snapshot Fields Reference

`collect()` returns a `Snapshot` dataclass with the following fields:

| Field | Type | Description |
|---|---|---|
| `cpu_per_core` | `list[float]` | Usage percentage for each logical core |
| `cpu_avg` | `float` | Average usage across all cores (%) |
| `cpu_freq_per_core` | `list[float]` | Current frequency per logical core (GHz); `[]` if unavailable |
| `mem_used` | `float` | Memory in use (GB) |
| `mem_total` | `float` | Total installed memory (GB) |
| `mem_percent` | `float` | Memory usage percentage |
| `disk_read_mbps` | `float` | Disk read throughput (MB/s) |
| `disk_write_mbps` | `float` | Disk write throughput (MB/s) |
| `net_recv_mbps` | `float` | Network receive throughput (MB/s, all interfaces) |
| `net_sent_mbps` | `float` | Network send throughput (MB/s, all interfaces) |
| `top_procs` | `list[dict]` | Process list; each entry contains `pid`, `name`, `cpu`, `mem_mb` |
| `timestamp` | `float` | Unix timestamp of the snapshot |

### 3.4 Filter Processes by Name


In [ ]:
# Show only processes whose name contains "python"
snap_py = collect(top_n=10, proc_filter=["python"])

if snap_py.top_procs:
    print(f"Found {len(snap_py.top_procs)} Python process(es):")
    for p in snap_py.top_procs:
        print(f"  PID {p['pid']:>6}  {p['name']:<20}  CPU {p['cpu']:.1f}%  MEM {p['mem_mb']:.0f} MB")
else:
    print("No processes matching 'python' are currently running")


### 3.5 Continuous Sampling — Monitor CPU for 10 Seconds


In [ ]:
import time
from pulsar import collect

# Warm up
collect()
time.sleep(0.3)

samples = []
n_samples = 10

print(f"Collecting {n_samples} CPU samples (one per second)...")
for i in range(n_samples):
    snap = collect()
    samples.append(snap.cpu_avg)
    print(f"  [{i+1:>2}/{n_samples}]  avg CPU: {snap.cpu_avg:5.1f}%  "
          f"mem: {snap.mem_percent:.1f}%  "
          f"disk R: {snap.disk_read_mbps:.2f} MB/s")
    time.sleep(1.0)

print(f"\n10-second summary:")
print(f"  Min CPU: {min(samples):.1f}%")
print(f"  Max CPU: {max(samples):.1f}%")
print(f"  Avg CPU: {sum(samples)/len(samples):.1f}%")


## 4. Command-Line Usage

Once installed, the `pulsar` command is available in any terminal. The examples below cover all common invocations.


### 4.1 Help


In [ ]:
!pulsar --help

### 4.2 `--once` Mode: Single Snapshot (table format)

Useful for scripting or a quick status check — prints one snapshot and exits immediately.


In [ ]:
!pulsar --once

### 4.3 `--once --format json`: Machine-Readable JSON Snapshot

Ideal for logging, data pipelines, or integration with other tools.


In [ ]:
!pulsar --once --format json

### 4.4 Parse a JSON Snapshot in Python


In [ ]:
import subprocess, json

result = subprocess.run(
    ["pulsar", "--once", "--format", "json"],
    capture_output=True, text=True
)
data = json.loads(result.stdout)

print("Top-level keys:", list(data.keys()))
print(f"\nTimestamp      : {data['timestamp']:.2f}")
print(f"CPU model      : {data['system']['cpu_model']}")
print(f"Avg CPU        : {data['cpu']['avg_percent']}%")
print(f"Memory usage   : {data['memory']['percent']}%")
print(f"Disk read      : {data['disk_io']['read_mbps']} MB/s")
print(f"Top processes  : {len(data['top_processes'])}")


### 4.5 `--top N`: Control the Number of Listed Processes


In [ ]:
# Show top 10 processes
!pulsar --once --top 10


### 4.6 `--proc NAME`: Filter Processes (repeatable)


In [ ]:
# Show only processes whose name contains "python" or "kernel"
!pulsar --once --proc python --proc kernel


### 4.7 `--interval`: Adjust the Refresh Rate

| Use case | Recommended interval |
|---|---|
| General monitoring | `1.0` (default) |
| High-frequency sampling | `0.5` or `0.25` |
| Low-power / background | `2.0` or `5.0` |

```bash
pulsar --interval 0.5   # refresh every 0.5 seconds
```

**Note:** In `--once` mode, `--interval` also controls the disk I/O warm-up wait (capped at 0.5 s).


### 4.8 Live Dashboard Mode

Run `pulsar` with no arguments to launch the interactive TUI dashboard (requires a real terminal — **cannot be run inside a notebook**):

```bash
# Basic launch
pulsar

# Fast refresh + show only python processes
pulsar --interval 0.5 --proc python

# Show top 10 processes
pulsar --top 10

# Rainbow disco mode
pulsar --disco
```

Dashboard keyboard shortcuts:

| Key | Action |
|---|---|
| `k` | Kill prompt (SIGTERM) — type a PID or process name, then Enter |
| `K` | Kill prompt (SIGKILL) — same as above, force-kill |
| `y` | Trigger fireworks animation |
| `q` or `Ctrl-C` | Quit |

In the kill prompt: **Esc** cancels without sending any signal.

> **Fun facts bar** — a rotating trivia panel sits at the bottom of the dashboard. Each fact is revealed character-by-character with a Pac-Man animation, then held for 10 seconds before fading out. The pool includes 500+ built-in facts across 22 categories, supplemented by live fetches from Wikipedia *On This Day* and random-facts APIs when an internet connection is available. Uptime milestone messages (at 5, 15, 30, 42, and 60 minutes) are also delivered through this bar.

## 5. Advanced: Timed CPU / Memory Logging


In [ ]:
import time, json
from datetime import datetime
from pulsar import collect

# Collect 5 samples one second apart and write them as JSON Lines
collect()          # warm up disk I/O baseline
time.sleep(0.3)

log_path = "/tmp/pulsar_log.jsonl"
with open(log_path, "w") as f:
    for _ in range(5):
        snap = collect(top_n=3)
        record = {
            "time": datetime.now().isoformat(),
            "cpu_avg": snap.cpu_avg,
            "mem_percent": snap.mem_percent,
            "disk_read_mbps": snap.disk_read_mbps,
            "disk_write_mbps": snap.disk_write_mbps,
        }
        f.write(json.dumps(record) + "\n")
        print(record)
        time.sleep(1.0)

print(f"\nLog saved to {log_path}")


## 6. Advanced: Visualise Collected Data with matplotlib


In [ ]:
import time
import matplotlib.pyplot as plt
from pulsar import collect

collect()
time.sleep(0.3)

timestamps = []
cpu_data   = []
mem_data   = []

print("Collecting 15 seconds of data...")
for i in range(15):
    snap = collect()
    timestamps.append(i)
    cpu_data.append(snap.cpu_avg)
    mem_data.append(snap.mem_percent)
    time.sleep(1.0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

ax1.plot(timestamps, cpu_data, color="steelblue", linewidth=2, marker="o", markersize=4)
ax1.set_ylabel("CPU Usage (%)")
ax1.set_ylim(0, 100)
ax1.axhline(80, color="orange", linestyle="--", linewidth=0.8, label="80% threshold")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("pulsar — 15-second hardware monitor")

ax2.plot(timestamps, mem_data, color="tomato", linewidth=2, marker="s", markersize=4)
ax2.set_ylabel("Memory Usage (%)")
ax2.set_xlabel("Time (s)")
ax2.set_ylim(0, 100)
ax2.axhline(90, color="orange", linestyle="--", linewidth=0.8, label="90% threshold")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/pulsar_plot.png", dpi=150)
plt.show()
print("Plot saved to /tmp/pulsar_plot.png")


## 7. Quick Reference

| Command | Description |
|---|---|
| `pulsar` | Launch the live TUI dashboard |
| `pulsar --once` | Print a single snapshot (table) and exit |
| `pulsar --once --format json` | Print a single snapshot (JSON) and exit |
| `pulsar --once --format json > snap.json` | Save snapshot to a file |
| `pulsar --interval 0.5` | Refresh every 0.5 seconds |
| `pulsar --top 10` | Show top 10 processes |
| `pulsar --proc python` | Filter to processes whose name contains `python` |
| `pulsar --proc python --proc node` | Filter by multiple process names |
| `pulsar --disco` | Rainbow disco mode |
| `pulsar --help` | Show help |

**Dashboard shortcuts:**
- `k` — kill prompt (SIGTERM): type PID or name, Enter to confirm, Esc to cancel
- `K` — kill prompt (SIGKILL): same as above, force-kill
- `y` — fireworks animation (also auto-triggers when any CPU core is pegged for 5 s)
- `q` / `Ctrl-C` — quit